https://github.com/CyanHuynh10/AI_EXERCISE

### 1. Phân tích P.E.A.S
* **P (Performance Measure):** Chỉ số hiệu suất dựa trên độ sạch của môi trường (ví dụ: +10 điểm cho mỗi ô được làm sạch) và tiết kiệm năng lượng (ví dụ: -1 điểm cho mỗi hành động di chuyển).
* **E (Environment):** Một lưới ô vuông (ví dụ 2x2 hoặc 3x3). Mỗi ô có thể ở trạng thái "Sạch" (Clean) hoặc "Bẩn" (Dirty). Môi trường có thể quan sát từng phần (Partially Observable) - Agent chỉ thấy ô hiện tại, nhưng nhờ Model nó sẽ ghi nhớ toàn bộ bản đồ.
* **A (Actuators):** Các bộ phận thực hiện hành động: Di chuyển (LÊN, XUỐNG, TRÁI, PHẢI), Hút bụi (SUCK), và Dừng (STOP).
* **S (Sensors):** Cảm biến vị trí (biết đang ở ô nào) và cảm biến bụi (biết ô hiện tại bẩn hay sạch).

### 2. Các thành phần của Agent theo kiến trúc Model-Based
* **State (Trạng thái hiện tại):** Là sự kết hợp giữa nhận thức (percept) hiện tại và bộ nhớ trong để hiểu về thế giới.
* **Percept:** Dữ liệu nhận vào gồm `[Vị trí, Tình trạng ô]`.
* **Model:**
    * **Hiểu biết về môi trường:** Agent hiểu rằng nếu thực hiện hành động "SUCK" thì ô đó sẽ sạch, nếu thực hiện "MOVE_RIGHT" thì tọa độ cột sẽ tăng lên.
    * **Lưu trữ thông tin (Internal State):** Một bản đồ nội bộ (Dictionary hoặc Matrix) lưu lại tình trạng của các ô đã đi qua. Điều này giúp Agent biết ô nào đã sạch mà không cần đứng tại đó.
* **Rules (Tập luật và Lập luận IF):**
    * `IF` trạng thái ô hiện tại là "Dirty" `THEN` Action = "SUCK".
    * `IF` tất cả các ô trong Model đều "Clean" `THEN` Action = "STOP".
    * `IF` ô hiện tại "Clean" `THEN` Tìm đường đến ô "Dirty" gần nhất dựa trên Model nội bộ.

### 3. Trạng thái môi trường (GOAL STATE)
Trạng thái đích đạt được khi tất cả các ô trong lưới môi trường đều chuyển về trạng thái **Clean**.
Ví dụ với lưới 3x3:
- (0,0): Clean
- (0,1): Clean
- (0,2): Clean
- (1,0): Clean
- (1,1): Clean
- (1,2): Clean
- (2,0): Clean
- (2,1): Clean
- (2,2): Clean

In [2]:
class VacuumEnvironment:
    def __init__(self, grid_size=(3, 3), dirty_cells=None):
        self.grid_size = grid_size
        self.grid = { (r, c): 'Clean' for r in range(grid_size[0]) for c in range(grid_size[1]) }
        if dirty_cells:
            for cell in dirty_cells:
                self.grid[cell] = 'Dirty'

    def get_percept(self, agent_pos):
        return {'pos': agent_pos, 'status': self.grid[agent_pos]}

    def execute_action(self, agent_pos, action):
        if action == 'SUCK':
            self.grid[agent_pos] = 'Clean'
        elif action == 'UP' and agent_pos[0] > 0:
            agent_pos = (agent_pos[0] - 1, agent_pos[1])
        elif action == 'DOWN' and agent_pos[0] < self.grid_size[0] - 1:
            agent_pos = (agent_pos[0] + 1, agent_pos[1])
        elif action == 'LEFT' and agent_pos[1] > 0:
            agent_pos = (agent_pos[0], agent_pos[1] - 1)
        elif action == 'RIGHT' and agent_pos[1] < self.grid_size[1] - 1:
            agent_pos = (agent_pos[0], agent_pos[1] + 1)
        return agent_pos

class ModelBasedVacuumAgent:
    def __init__(self, grid_size):
        self.grid_size = grid_size
        self.model = { (r, c): 'Unknown' for r in range(grid_size[0]) for c in range(grid_size[1]) }
        
    def update_model(self, pos, status):
        self.model[pos] = status

    def get_heuristic(self):
        return sum(1 for status in self.model.values() if status == 'Dirty' or status == 'Unknown')

    def choose_action(self, percept):
        pos = percept['pos']
        status = percept['status']
        
        self.update_model(pos, status)
        h = self.get_heuristic()
        
        if status == 'Dirty':
            return 'SUCK', h
        
        for r in range(self.grid_size[0]):
            for c in range(self.grid_size[1]):
                if self.model[(r, c)] != 'Clean':
                    if r < pos[0]: return 'UP', h
                    if r > pos[0]: return 'DOWN', h
                    if c < pos[1]: return 'LEFT', h
                    if c > pos[1]: return 'RIGHT', h
        
        return 'STOP', 0

def print_board(grid, agent_pos):
    print("+-----+-----+-----+")
    for r in range(3):
        row_str = "|"
        for c in range(3):
            status = 'D' if grid[(r, c)] == 'Dirty' else 'C'
            if (r, c) == agent_pos:
                val_str = f" [{status}] " 
            else:
                val_str = f"  {status}  "
            row_str += val_str + "|"
        print(row_str)
        print("+-----+-----+-----+")

def run_simulation():
    env = VacuumEnvironment(grid_size=(3, 3), dirty_cells=[(0, 2), (1, 1), (2, 0), (2, 2)])
    agent = ModelBasedVacuumAgent(grid_size=(3, 3))
    agent_pos = (0, 0)
    step = 0
    
    percept = env.get_percept(agent_pos)
    agent.update_model(percept['pos'], percept['status'])
    initial_h = agent.get_heuristic()

    print("=========================================================")
    print(f"TRẠNG THÁI BAN ĐẦU (Step: {step} | Heuristic: {initial_h})")
    print("=========================================================")
    print_board(env.grid, agent_pos)

    while True:
        step += 1
        percept = env.get_percept(agent_pos)
        action, h = agent.choose_action(percept)
        
        if action == 'STOP':
            print("=========================================================")
            print(f"KẾT QUẢ: MÔI TRƯỜNG ĐÃ SẠCH HOÀN TOÀN TRONG {step - 1} BƯỚC!")
            print("=========================================================")
            break
            
        agent_pos = env.execute_action(agent_pos, action)
        
        print("---------------------------------------------------------")
        print(f"Step: {step:02d} | Action: {action} | Heuristic: {h}")
        print_board(env.grid, agent_pos)
        
        if step > 50:
            print("Simulation timed out!")
            break

run_simulation()

TRẠNG THÁI BAN ĐẦU (Step: 0 | Heuristic: 8)
+-----+-----+-----+
| [C] |  C  |  D  |
+-----+-----+-----+
|  C  |  D  |  C  |
+-----+-----+-----+
|  D  |  C  |  D  |
+-----+-----+-----+
---------------------------------------------------------
Step: 01 | Action: RIGHT | Heuristic: 8
+-----+-----+-----+
|  C  | [C] |  D  |
+-----+-----+-----+
|  C  |  D  |  C  |
+-----+-----+-----+
|  D  |  C  |  D  |
+-----+-----+-----+
---------------------------------------------------------
Step: 02 | Action: RIGHT | Heuristic: 7
+-----+-----+-----+
|  C  |  C  | [D] |
+-----+-----+-----+
|  C  |  D  |  C  |
+-----+-----+-----+
|  D  |  C  |  D  |
+-----+-----+-----+
---------------------------------------------------------
Step: 03 | Action: SUCK | Heuristic: 7
+-----+-----+-----+
|  C  |  C  | [C] |
+-----+-----+-----+
|  C  |  D  |  C  |
+-----+-----+-----+
|  D  |  C  |  D  |
+-----+-----+-----+
---------------------------------------------------------
Step: 04 | Action: DOWN | Heuristic: 6
+-----